In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from pathlib import Path


In [3]:
BASE_DIR = Path("..")

for path in BASE_DIR.iterdir():
    print(path)

..\.gitignore
..\api
..\artifacts
..\config.yaml
..\Data
..\docker-compose.yml
..\Dockerfile
..\mlruns
..\models
..\notebooks
..\README.md
..\requirements.txt
..\src
..\test
..\venv


In [4]:
DATA_DIR = Path("../Data")

for path in DATA_DIR.rglob("*"):
    print(path)

..\Data\Processed
..\Data\Raw
..\Data\Processed\creditcard_clean.csv
..\Data\Processed\X_test_scaled.csv
..\Data\Processed\X_train_smote.csv
..\Data\Processed\y_test.csv
..\Data\Processed\y_train_smote.csv
..\Data\Raw\creditcard.csv


In [5]:
X_train = pd.read_csv("../Data/Processed/X_train_smote.csv")
y_train = pd.read_csv("../Data/Processed/y_train_smote.csv")

X_test = pd.read_csv("../Data/Processed/X_test_scaled.csv")
y_test = pd.read_csv("../Data/Processed/y_test.csv")

In [6]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (453204, 32)
y_train: (453204, 1)
X_test: (56746, 32)
y_test: (56746, 1)


In [7]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nTraining target percentage:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTest target distribution:")
print(y_test.value_counts())

print("\nTest target percentage:")
print(y_test.value_counts(normalize=True) * 100)

Training target distribution:
Class
0        226602
1        226602
Name: count, dtype: int64

Training target percentage:
Class
0        50.0
1        50.0
Name: proportion, dtype: float64

Test target distribution:
Class
0        56651
1           95
Name: count, dtype: int64

Test target percentage:
Class
0        99.832587
1         0.167413
Name: proportion, dtype: float64


In [8]:
y_train = y_train.squeeze()
y_test = y_test.squeeze()

In [9]:
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

print("\ny_train type:", type(y_train))
print("y_test type:", type(y_test))

y_train shape: (453204,)
y_test shape: (56746,)

y_train type: <class 'pandas.core.series.Series'>
y_test type: <class 'pandas.core.series.Series'>


In [10]:
print("Unique classes:", y_train.unique())
print("Train classes:", y_train.value_counts().to_dict())
print("Test classes:", y_test.value_counts().to_dict())

Unique classes: [0 1]
Train classes: {0: 226602, 1: 226602}
Test classes: {0: 56651, 1: 95}


In [11]:
##Baseline model
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

In [12]:
log_reg.fit(X_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [13]:
#Generate the predictions
y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:,1]

In [14]:
def evaluate_model(model_name, y_true, y_pred, y_pred_proba):
    print(f"===== {model_name} =====")
    
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_true, y_pred_proba):.4f}")
    print(f"PR-AUC:    {average_precision_score(y_true, y_pred_proba):.4f}")
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

In [15]:
evaluate_model(
    "Logistic Regression",
    y_test,
    y_pred,
    y_pred_proba
)

===== Logistic Regression =====
Precision: 0.0666
Recall:    0.8737
F1 Score:  0.1237
ROC-AUC:   0.9641
PR-AUC:    0.6862

Confusion Matrix:
[[55487  1164]
 [   12    83]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56651
           1       0.07      0.87      0.12        95

    accuracy                           0.98     56746
   macro avg       0.53      0.93      0.56     56746
weighted avg       1.00      0.98      0.99     56746



The baseline demonstrates that the available features contain substantial predictive signal. Logistic Regression achieves strong discrimination and high fraud recall, but the default 0.5 decision threshold results in poor precision and a high false-positive burden. This provides a strong baseline for investigating whether other models and threshold optimization can improve the precision-recall trade-off

Our Logistic Regression baseline achieves a ROC-AUC of 0.964 and PR-AUC of 0.686, indicating strong discriminatory performance despite severe class imbalance. At the default threshold, it recalls 87.4% of fraud cases, but precision is only 6.7%, with 1,164 false positives versus 83 true positives. So the model detects most fraud but produces too many false alerts, giving us a clear precision-recall trade-off to investigate with alternative models and threshold optimization

In [16]:
## Baseline model No.2
from sklearn.ensemble import RandomForestClassifier


In [17]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [18]:
rf.fit(X_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [19]:
rf_pred = rf.predict(X_test)
rf_pred_proba = rf.predict_proba(X_test)[:,1]

In [20]:
evaluate_model(
    "Random Forest",
    y_test,
    rf_pred,
    rf_pred_proba
)

===== Random Forest =====
Precision: 0.8987
Recall:    0.7474
F1 Score:  0.8161
ROC-AUC:   0.9669
PR-AUC:    0.8113

Confusion Matrix:
[[56643     8]
 [   24    71]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.90      0.75      0.82        95

    accuracy                           1.00     56746
   macro avg       0.95      0.87      0.91     56746
weighted avg       1.00      1.00      1.00     56746



1. Confusion matrix:
   71 fraud cases are actualy correct
   the model missed 24 fraudulent transections
   it incorrectly flagged 8 legit transection as fraud

   The Random Forest produces only 8 false fraud alerts while detecting 71 of the 95 fraud cases. It misses 24 fraud cases.

2. Recall:
   0.7474/74.74%

   The model detects 71 of the 95 fraudulent transactions, meaning it misses 24 fraud cases.
   Random Forest sacrifices recall relative to Logistic Regression, detecting 12 fewer fraud cases.

3. Precision:
   0.8987/ 89.87%
   Logistic Regression catches more fraud, but produces 1,164 false alarms.

   Random Forest catches less fraud, but produces only 8 false alarms.


4. F1:
   The balance is significantly higher than the prev model
   Logistic Regression F1 = 0.1237
   Random Forest F1      = 0.8161

   So Logistic Regression essentially chose:
   Catch lots of fraud, even if I generate enormous numbers of false alarms.

  Random Forest behaves more like:
  Only raise an alert when I'm much more confident, accepting that I'll miss more fraud.

5. PR-AUC:
   overall behavior improved


6 ROC-AUC:
  Both models demonstrate strong ranking/discrimination ability, and Random Forest has a modestly higher ROC-AUC. The much more notable difference is in PR-AUC and the operating-point precision/recall trade-off.

7. Camparison of the two models:
   Random Forest substantially changes the operating characteristics of the baseline. Compared with Logistic Regression, it reduces false positives from 1,164 to only 8 and increases precision from 6.66% to 89.87%, while recall decreases from 87.37% to 74.74%. This means the Random Forest produces far fewer false alerts but misses twice as many fraud cases. Its F1 and PR-AUC are substantially higher, while the ROC-AUC improvement is relatively small. Therefore, the primary improvement isn't simply better discrimination; it's a much more favorable precision-recall profile at the evaluated threshold.

In [21]:
baseline_results = []

In [22]:
def get_confusion_counts(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    return {
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

In [23]:
log_cm = get_confusion_counts(y_test, y_pred)
rf_cm = get_confusion_counts(y_test, rf_pred)

In [24]:
baseline_results.append({
    "Model": "Logistic Regression",
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_pred_proba),
    "PR-AUC": average_precision_score(y_test, y_pred_proba),
    **log_cm
})

In [25]:
baseline_results.append({
    "Model": "Random Forest",
    "Precision": precision_score(y_test, rf_pred),
    "Recall": recall_score(y_test, rf_pred),
    "F1": f1_score(y_test, rf_pred),
    "ROC-AUC": roc_auc_score(y_test, rf_pred_proba),
    "PR-AUC": average_precision_score(y_test, rf_pred_proba),
    **rf_cm
})

In [26]:
baseline_results_df = pd.DataFrame(baseline_results)

baseline_results_df

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC,TN,FP,FN,TP
0,Logistic Regression,0.066560,0.873684,0.123696,0.964081,0.686204,55487,1164,12,83
1,Random Forest,0.898734,0.747368,0.816092,0.966881,0.811302,56643,8,24,71


In [27]:
from xgboost import XGBClassifier

In [28]:
xgb = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric="logloss"
)

In [29]:
xgb.fit(X_train,y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [30]:
xgb_pred = xgb.predict(X_test)
xgb_pred_proba= xgb.predict_proba(X_test)[:,1]

In [31]:
evaluate_model(
    "XGB Boost",
    y_test,
    xgb_pred,
    xgb_pred_proba
)

===== XGB Boost =====
Precision: 0.8488
Recall:    0.7684
F1 Score:  0.8066
ROC-AUC:   0.9691
PR-AUC:    0.8085

Confusion Matrix:
[[56638    13]
 [   22    73]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.85      0.77      0.81        95

    accuracy                           1.00     56746
   macro avg       0.92      0.88      0.90     56746
weighted avg       1.00      1.00      1.00     56746



In [32]:
xgb_cm= get_confusion_counts(y_test,xgb_pred)

In [35]:
baseline_results.append({
    "Model": "XGBoost",
    "Precision": precision_score(y_test, xgb_pred),
    "Recall": recall_score(y_test, xgb_pred),
    "F1": f1_score(y_test, xgb_pred),
    "ROC-AUC": roc_auc_score(y_test, xgb_pred_proba),
    "PR-AUC": average_precision_score(y_test, xgb_pred_proba),
    **xgb_cm
})

In [39]:
baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df

,Model,Precision,Recall,F1,ROC-AUC,PR-AUC,TN,FP,FN,TP
0,Logistic Regression,0.066560,0.873684,0.123696,0.964081,0.686204,55487,1164,12,83
1,Random Forest,0.898734,0.747368,0.816092,0.966881,0.811302,56643,8,24,71
2,XGBoost,0.848837,0.768421,0.806630,0.969129,0.808535,56638,13,22,73
3,XGBoost,0.848837,0.768421,0.806630,0.969129,0.808535,56638,13,22,73
